In [7]:



import json
import pandas as pd
import numpy as np
from collections import Counter
from pathlib import Path
from inspect_ai.log import read_eval_log
import matplotlib.pyplot as plt

In [5]:
LOG_DIR = "./logs"
TASK_NAME = "listing_generator_eval"

RUN_PIPELINE = True 

In [ ]:
"""
Run Full Pipeline
"""
from property_listing.eval.pipeline import run_eval

evaluation_results = run_eval()

print(evaluation_results[0].location)




In [20]:
# log_path = sorted(Path(LOG_DIR).glob("*.eval"))[-1]
log_path = "./logs/2026-06-15T13-48-18-00-00_listing-generator-eval_kVS3m5AS6wxU9VpHB5p9bv.eval"

In [ ]:
log = read_eval_log(str(log_path))
rows = []

for s in log.samples:
    base = {
        "id": s.id,
        "property": (s.metadata or {}).get("property"),
        "output": s.metadata["generated_copy_json"],
    }

    scores = getattr(s, "scores", {}) or {}

    for metric_name, score in scores.items():
        rows.append({
            **base,
            "metric": metric_name,
            "value": score.value,
            "explanation": score.explanation,
            "audit": (score.metadata or {}).get("audit_checks", [])
        })

df = pd.DataFrame(rows)
df.head()


{'slug_headline': "The Luminary Penthouse: Panoramic Skyline Views & Private Terrace in Chicago's Cultural Heart", 'hero_paragraph': "Experience the ultimate urban retreat in this architectural masterpiece, The Luminary Penthouse. Featuring double-height ceilings and floor-to-ceiling windows, this stunning loft floods with natural light and offers breathtaking panoramic skyline views. Located just steps away from Chicago's vibrant cultural district, you'll enjoy unparalleled access to fine dining and convenient transit options.", 'amenity_highlights': ['Blazing-fast High-Speed Internet', 'Convenient Secured Parking', 'Comfortable Central Air Conditioning'], 'guest_expectations': 'Check-in is at 3 PM, and check-out is by 11 AM. Enjoy peace of mind with a full refund available up to 5 days prior to arrival; a 50% deposit is due at booking, with the remaining balance 7 days before check-in. A $300 damage deposit hold on your card is required at check-in.'}
{'slug_headline': 'Serenity Bay:

,id,property,output,metric,value,explanation,audit
0,10482,"{'property_id': 10482, 'property_name': 'The L...",,groundedness_scorer_cot,0.7000,Groundedness Score: 0.70 (14/20 claims verifie...,"[{'claim': 'The Luminary Penthouse', 'verdict'..."
1,10483,"{'property_id': 10483, 'property_name': 'Seren...",,groundedness_scorer_cot,0.9375,Groundedness Score: 0.94 (15/16 claims verifie...,"[{'claim': 'Property name is 'Serenity Bay'.',..."


In [8]:
import pandas as pd

rows = []
for s in log.samples:
    rows.append({
        "id": s.id,
        "score": getattr(s.score, "value", None),
        "explanation": getattr(s.score, "explanation", None),
        "output": getattr(s.output, "text", None),
        "metadata": getattr(s.score, "metadata", None),
    })

df = pd.DataFrame(rows)
df.head()

The 'score' field is deprecated. Access sample scores through 'scores' instead.


,id,score,explanation,output,metadata
0,10482,0.7000,Groundedness Score: 0.70 (14/20 claims verifie...,None,{'audit_checks': [{'claim': 'The Luminary Pent...
1,10483,0.9375,Groundedness Score: 0.94 (15/16 claims verifie...,None,{'audit_checks': [{'claim': 'Property name is ...


In [ ]:
def expand_metadata(df):
    meta = df["metadata"].apply(lambda x: x if isinstance(x, dict) else {})
    meta_df = pd.json_normalize(meta)
    return pd.concat([df.drop(columns=["metadata"]), meta_df], axis=1)

df = expand_metadata(df)
df.head()

In [ ]:
summary = {
    "num_samples": len(df),
    "mean_score": df["score"].mean(),
    "min_score": df["score"].min(),
    "max_score": df["score"].max(),
}

summary

In [ ]:
plt.figure()
df["score"].hist(bins=10)
plt.title("Score Distribution")
plt.xlabel("Score")
plt.ylabel("Frequency")
plt.show()

In [ ]:
low = df[df["score"] < df["score"].quantile(0.25)]
low[["sample_id", "score", "explanation"]].head(10)

In [ ]:
if "total_claims" in df.columns:
    df["grounded_ratio"] = df["score"]
    
    plt.figure()
    df["grounded_ratio"].hist(bins=10)
    plt.title("Groundedness Ratio Distribution")
    plt.xlabel("Grounded Ratio")
    plt.ylabel("Frequency")
    plt.show()

In [ ]:
def extract_failed_claims(row):
    meta = row.get("metadata", {})
    if isinstance(meta, dict):
        checks = meta.get("audit_checks", [])
        return [c for c in checks if not c.get("verdict")]
    return []

df["failed_claims"] = df.apply(extract_failed_claims, axis=1)

In [ ]:
from collections import Counter

all_failed = []
for item in df["failed_claims"]:
    for c in item:
        all_failed.append(c.get("claim"))

Counter(all_failed).most_common(20)

In [ ]:
def view_sample(sample_id):
    row = df[df["sample_id"] == sample_id].iloc[0]
    
    print("SAMPLE ID:", sample_id)
    print("\nSCORE:", row["score"])
    print("\nEXPLANATION:\n", row["explanation"])
    print("\nOUTPUT:\n", row.get("output"))
    
# Example:
view_sample(df["sample_id"].iloc[0])